# Paper 1A — three GSEM robustness tables, plus a number audit

The V1 manuscript (`Paper15_ 1A_Sealed_Window_TPB_V1.pdf`) reports the primary Mundlak GSEM on the full analytic sample. An earlier plan item (**A3**) also claimed three robustness checks that were never written to disk:

1. **R1 — drop COVID cohorts.** Exclude guests whose first review falls in 2020 or 2021.
2. **R2 — thirty largest destinations.** Restrict to the 30 cities with the largest analytic *N* (same ranking as `tableA1_top_cities.csv`).
3. **R3 — destination fixed effects.** On a random subsample of 150,000 guests (seed `20260901`), replace Mundlak city means with city dummies.

This notebook estimates the same two-equation specification as `src/p1_step5_gsem.py` (Gaussian BI, logit behaviour, year dummies, city-clustered SE, construct scores `z(log1p(hits))`) under each restriction, writes CSV tables to `paper1A/tables/`, and then audits every number in the V1 PDF against `outputs/tables/*.csv`.


In [ ]:
import os, sys, time
import pandas as pd
from IPython.display import display, Markdown

ROOT = os.path.abspath(os.path.join(os.getcwd() if os.path.basename(os.getcwd()) != 'paper1A' else os.path.join(os.getcwd(), '..')))
# Prefer the project root regardless of kernel cwd
for cand in [
    os.getcwd(),
    os.path.abspath(os.path.join(os.getcwd(), '..')),
    r'D:/Support/Paper2026/15_BehaviorTheory_ML/Paper1',
]:
    if os.path.isfile(os.path.join(cand, 'work', 'analysis.parquet')):
        ROOT = cand
        break
P1A = os.path.join(ROOT, 'paper1A')
sys.path.insert(0, P1A)
print('ROOT =', ROOT)
print('P1A  =', P1A)


## 1. Analytic sample and the three cuts


In [ ]:
import numpy as np
df = pd.read_parquet(os.path.join(ROOT, 'work', 'analysis.parquet'))
print(f'Full analytic N = {len(df):,}  cities = {df.city.nunique()}')

covid = df[~df['year'].isin([2020, 2021])].copy()
top30 = df.groupby('city').size().sort_values(ascending=False).head(30).index
big = df[df['city'].isin(top30)].copy()
print(f'R1 exclude 2020–2021: N = {len(covid):,}  cities = {covid.city.nunique()}  years dropped = 101,577')
print(f'R2 thirty largest destinations: N = {len(big):,}  cities = {big.city.nunique()}')
print('Top 30:')
print(df.groupby('city').size().sort_values(ascending=False).head(30).to_string())


## 2. Estimate the three specifications

The helper `p1a_robustness_gsem.py` reuses the primary count specification. R3 draws `n = 150,000` with `numpy` generator seed `20260901`, drops Mundlak means, adds city dummies, and drops cities with no outcome variation (complete separation under logit FE). Each spec is estimated twice (Y1 same-listing return, Y2 platform continuance).

Re-running takes roughly 15–25 minutes. If the CSVs already exist, skip estimation and load them.


In [ ]:
from p1a_robustness_gsem import main as run_robust, OUT, TAB

summary_path = os.path.join(OUT, 'table_robust_gsem_summary.csv')
FORCE = False  # set True to re-estimate even if tables exist

if FORCE or not os.path.isfile(summary_path):
    t0 = time.time()
    summary = run_robust()
    print(f'finished in {time.time()-t0:.0f}s')
else:
    summary = pd.read_csv(summary_path)
    print('loaded existing', summary_path)

display(summary)


## 3. Coefficient tables (same layout as manuscript Tables 3–4)

Each file has four columns: BI equation, Y1 behaviour equation, BI equation (repeated), Y2 behaviour equation. Mundlak city means and year (and, in R3, city) dummies are included and suppressed, matching the note under Tables 3–4.


In [ ]:
from pathlib import Path

def show(name, caption):
    p = os.path.join(OUT, name)
    display(Markdown(f'### {caption}'))
    display(Markdown(f'`{p}`'))
    t = pd.read_csv(p)
    display(t)
    return t

t_covid = show('table_robust_covid.csv', 'R1 — exclude 2020–2021 first-review cohorts')
t_top30 = show('table_robust_top30.csv', 'R2 — thirty largest destinations')
t_fe    = show('table_robust_cityfe.csv', 'R3 — destination FE on n = 150,000 (no Mundlak)')


## 4. Key-path comparison against the published Mundlak baseline

Baseline coefficients are read from `table_gsem_revisit.csv` / `table_gsem_continue.csv` (not re-estimated). The intention→behaviour path is the quantity Paper 1A actually claims; the antecedent paths are reported for completeness and should not be read as identified (see manuscript §5.4).


In [ ]:
def grab(tbl, variable, col):
    row = tbl.loc[tbl['variable'] == variable]
    if row.empty:
        return ''
    return str(row[col].iloc[0])

base_y1 = pd.read_csv(os.path.join(ROOT, 'outputs', 'tables', 'table_gsem_revisit.csv'))
base_y2 = pd.read_csv(os.path.join(ROOT, 'outputs', 'tables', 'table_gsem_continue.csv'))

keys = [
    'Revisit intention (BI)',
    'Attitude',
    'Subjective norm',
    'Perceived behavioural control',
    'Satisfaction',
]
cmp_rows = []
for v in keys:
    cmp_rows.append({
        'path': v,
        'baseline_Y1': grab(base_y1, v, 'eq2_behaviour'),
        'R1_covid_Y1': grab(t_covid, v, 'eq2_revisit'),
        'R2_top30_Y1': grab(t_top30, v, 'eq2_revisit'),
        'R3_cityFE_Y1': grab(t_fe, v, 'eq2_revisit'),
        'baseline_Y2': grab(base_y2, v, 'eq2_behaviour'),
        'R1_covid_Y2': grab(t_covid, v, 'eq2_continue'),
        'R2_top30_Y2': grab(t_top30, v, 'eq2_continue'),
        'R3_cityFE_Y2': grab(t_fe, v, 'eq2_continue'),
    })
cmp = pd.DataFrame(cmp_rows)
cmp_path = os.path.join(OUT, 'table_robust_vs_baseline.csv')
cmp.to_csv(cmp_path, index=False)
cmp.to_csv(os.path.join(ROOT, 'outputs', 'tables', 'table_robust_vs_baseline.csv'), index=False)
display(Markdown('**Behaviour equation (eq2), construct paths**'))
display(cmp)
print('wrote', cmp_path)


## 5. Number audit of the V1 manuscript against the CSV sources

Every tabulated number in `Paper15_ 1A_Sealed_Window_TPB_V1.pdf`, plus the derived claims in the abstract and §4, is compared with `outputs/tables/*.csv` and (for Table 1 sample counts) with `work/analysis.parquet`. Status codes:

- **match** — identical
- **rounding** — paper rounded (0.234 vs 0.2339); acceptable
- **mismatch** — paper value is not the current CSV
- **wording** — the number is defensible under one reading but the sentence is imprecise


In [ ]:
from p1a_number_audit import main as run_audit
audit = run_audit()
display(audit['status'].value_counts().rename('n').to_frame())
bad = audit[audit['status'].isin(['mismatch', 'wording'])]
display(Markdown('### Items that need an author decision'))
display(bad)
display(Markdown('### Full audit trail'))
display(audit)


## Files written

| File | Content |
|---|---|
| `paper1A/tables/table_robust_covid.csv` | R1, both outcomes |
| `paper1A/tables/table_robust_top30.csv` | R2, both outcomes |
| `paper1A/tables/table_robust_cityfe.csv` | R3, both outcomes |
| `paper1A/tables/table_robust_gsem_summary.csv` | N, cities, BI→Y for all six fits |
| `paper1A/tables/table_robust_vs_baseline.csv` | Side-by-side with Tables 3–4 |
| `paper1A/tables/table_number_audit_1A.csv` | Full number audit |
| `paper1A/tables/table_robust_*_{revisit,continue}.csv` | Single-outcome layout matching Tables 3–4 |

Copies are also written to `outputs/tables/` so `paper/build_1A.js` can pick them up.
